# 04 — Validate ohsome API responses on the 10-cell test sample

This notebook performs real requests against the current ohsome API for the deterministic 10-cell sample prepared in Notebook 03.

It is deliberately a validation notebook. It:
1. records `/metadata` and the OSHDB temporal extent;
2. queries road length and current road-element count at the latest OSHDB snapshot;
3. queries unique users and contributions over the full OSHDB history;
4. persists every successful response and every HTTP error body;
5. validates endpoint-specific response schemas before parsing;
6. produces one tidy 10-cell validation table keyed by `grid_1km_id`.

No request is made for the full set of 895 cells in this notebook.


In [1]:
from pathlib import Path
import json
import time
from datetime import datetime, timezone
import pandas as pd
import geopandas as gpd
import requests

API_BASE = "https://api.ohsome.org/v1"
ROAD_FILTER = "highway=* and geometry:line"
REQUEST_TIMEOUT = 180
TRANSIENT_STATUS = {429, 500, 502, 503, 504}

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, start.parent, *start.parents]:
        if (candidate / "notebooks").exists() and (candidate / "Petropolis").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root.")

PROJECT_ROOT = find_project_root(Path.cwd())
INPUT_DIR = PROJECT_ROOT / "data" / "intermediate" / "ohsome" / "Petropolis"
TEST_BOUNDARIES_FILE = INPUT_DIR / "petropolis_grid_1km_ohsome_test_boundaries.geojson"
TEST_INDEX_FILE = INPUT_DIR / "petropolis_grid_1km_ohsome_test_cells.parquet"
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "ohsome" / "Petropolis" / "test_10_cells"
RESULTS_DIR = PROJECT_ROOT / "data" / "processed" / "Petropolis"
RAW_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_FILE = RESULTS_DIR / "petropolis_ohsome_test_10_cells.parquet"

for p in (TEST_BOUNDARIES_FILE, TEST_INDEX_FILE):
    if not p.exists():
        raise FileNotFoundError(f"Required Notebook 03 output not found: {p}")

session = requests.Session()
session.headers.update({"User-Agent":"sotm-br-2026-research/0.2 (academic OSM analysis)"})


## 1. Read `/metadata` and define temporal parameters


In [2]:
r = session.get(f"{API_BASE}/metadata", timeout=REQUEST_TIMEOUT)
r.raise_for_status()
metadata = r.json()
(RAW_DIR / "ohsome_metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")

api_version = metadata.get("apiVersion")
temporal = metadata.get("extractRegion", {}).get("temporalExtent", {})
oshdb_start = temporal.get("fromTimestamp")
oshdb_end = temporal.get("toTimestamp")
if not oshdb_start or not oshdb_end:
    raise ValueError("Could not read extractRegion.temporalExtent from /metadata.")

users_time = f"{oshdb_start}/{oshdb_end}"
contributions_time = f"{oshdb_start},{oshdb_end}"

print("API version:", api_version)
print("OSHDB start:", oshdb_start)
print("OSHDB end:", oshdb_end)
print("Users interval:", users_time)
print("Contributions interval:", contributions_time)


API version: 1.10.4
OSHDB start: 2007-10-08T00:00:00Z
OSHDB end: 2026-07-27T09:00Z
Users interval: 2007-10-08T00:00:00Z/2026-07-27T09:00Z
Contributions interval: 2007-10-08T00:00:00Z,2026-07-27T09:00Z


## 2. Load and validate the 10 identified boundaries


In [3]:
test_cells = gpd.read_parquet(TEST_INDEX_FILE)
feature_collection = json.loads(TEST_BOUNDARIES_FILE.read_text(encoding="utf-8"))
features = feature_collection.get("features", [])
ids = [str(f.get("properties", {}).get("id")) for f in features]

checks = {
    "ten_cells": len(test_cells) == 10,
    "ten_features": len(features) == 10,
    "ids_unique": len(ids) == len(set(ids)),
    "ids_match": set(ids) == set(test_cells["grid_1km_id"].astype(str)),
}
print(checks)
if not all(checks.values()):
    raise ValueError(f"Boundary validation failed: {checks}")

bpolys = json.dumps(feature_collection, ensure_ascii=False, separators=(",", ":"))


{'ten_cells': True, 'ten_features': True, 'ids_unique': True, 'ids_match': True}


## 3. HTTP helper with raw success/error persistence


In [4]:
def ohsome_post(endpoint, data, stem, max_attempts=4):
    url = f"{API_BASE}{endpoint}"
    last = None
    for attempt in range(1, max_attempts + 1):
        response = session.post(url, data=data, timeout=REQUEST_TIMEOUT)
        content_type = response.headers.get("content-type", "")
        try:
            body = response.json()
        except Exception:
            body = {"raw_text": response.text}

        envelope = {
            "request": {
                "endpoint": endpoint,
                "data_without_bpolys": {k:v for k,v in data.items() if k != "bpolys"},
                "bpolys_feature_count": len(features),
            },
            "response": {
                "status_code": response.status_code,
                "content_type": content_type,
                "body": body,
            },
            "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        suffix = "json" if response.ok else "error.json"
        (RAW_DIR / f"{stem}.{suffix}").write_text(
            json.dumps(envelope, indent=2, ensure_ascii=False), encoding="utf-8"
        )

        if response.ok:
            return body

        last = RuntimeError(
            f"{endpoint} returned HTTP {response.status_code}: "
            f"{json.dumps(body, ensure_ascii=False)[:1200]}"
        )
        if response.status_code not in TRANSIENT_STATUS or attempt == max_attempts:
            raise last
        time.sleep(min(2 ** attempt, 20))
    raise last


## 4. Execute the four endpoint tests


In [5]:
common = {"bpolys": bpolys, "filter": ROAD_FILTER, "format": "json"}

payloads = {}
payloads["road_length_m"] = ohsome_post(
    "/elements/length/groupBy/boundary",
    common,
    "elements_length_groupby_boundary",
)
payloads["road_element_count"] = ohsome_post(
    "/elements/count/groupBy/boundary",
    common,
    "elements_count_groupby_boundary",
)
payloads["contributors_count"] = ohsome_post(
    "/users/count/groupBy/boundary",
    {**common, "time": users_time},
    "users_count_groupby_boundary",
)
payloads["contributions_count"] = ohsome_post(
    "/contributions/count/groupBy/boundary",
    {**common, "time": contributions_time},
    "contributions_count_groupby_boundary",
)
print("All four HTTP requests returned success.")


All four HTTP requests returned success.


## 5. Inspect and validate the real response schemas


In [6]:
def endpoint_schema(payload):
    groups = payload.get("groupByResult")
    if not isinstance(groups, list):
        return {"groupByResult_list": False}
    sample = None
    for g in groups:
        result = g.get("result")
        if isinstance(result, list) and result:
            sample = result[0]
            break
    return {
        "groupByResult_list": True,
        "groups": len(groups),
        "sample_result_keys": sorted(sample.keys()) if isinstance(sample, dict) else [],
        "apiVersion": payload.get("apiVersion"),
    }

schema_report = {name: endpoint_schema(p) for name,p in payloads.items()}
display(pd.DataFrame(schema_report).T)

expected_ids = set(test_cells["grid_1km_id"].astype(str))
for name, payload in payloads.items():
    groups = payload.get("groupByResult")
    if not isinstance(groups, list):
        raise ValueError(f"{name}: missing groupByResult list.")
    returned_ids = {str(g.get("groupByObject")) for g in groups}
    if returned_ids != expected_ids:
        raise ValueError(
            f"{name}: boundary IDs differ from test sample. "
            f"missing={sorted(expected_ids-returned_ids)}, extra={sorted(returned_ids-expected_ids)}"
        )


,groupByResult_list,groups,sample_result_keys,apiVersion
road_length_m,True,10,"[timestamp, value]",1.10.4
road_element_count,True,10,"[timestamp, value]",1.10.4
contributors_count,True,10,"[fromTimestamp, toTimestamp, value]",1.10.4
contributions_count,True,10,"[fromTimestamp, toTimestamp, value]",1.10.4


## 6. Endpoint-specific parser


In [7]:
def parse_grouped(payload, metric, temporal_kind):
    rows = []
    for group in payload["groupByResult"]:
        gid = str(group.get("groupByObject"))
        result = group.get("result")
        if not isinstance(result, list) or len(result) != 1:
            raise ValueError(f"{metric}/{gid}: expected exactly one result interval, got {result}")
        item = result[0]
        if "value" not in item:
            raise ValueError(f"{metric}/{gid}: missing value.")
        row = {"grid_1km_id": gid, metric: float(item["value"])}
        if temporal_kind == "snapshot":
            if "timestamp" not in item:
                raise ValueError(f"{metric}/{gid}: expected timestamp.")
            row[f"{metric}_timestamp"] = item["timestamp"]
        elif temporal_kind == "interval":
            if "fromTimestamp" in item and "toTimestamp" in item:
                row[f"{metric}_from"] = item["fromTimestamp"]
                row[f"{metric}_to"] = item["toTimestamp"]
            elif "timestamp" in item:
                row[f"{metric}_timestamp"] = item["timestamp"]
            else:
                raise ValueError(f"{metric}/{gid}: no recognized temporal fields: {item.keys()}")
        rows.append(row)
    return pd.DataFrame(rows)

length_df = parse_grouped(payloads["road_length_m"], "road_length_m", "snapshot")
elements_df = parse_grouped(payloads["road_element_count"], "road_element_count", "snapshot")
users_df = parse_grouped(payloads["contributors_count"], "contributors_count", "interval")
contrib_df = parse_grouped(payloads["contributions_count"], "contributions_count", "interval")


## 7. Consolidate the 10-cell test result and quality-control it


In [8]:
base_cols = [
    "grid_1km_id", "cell_area_m2", "municipality_fraction",
    "spatial_relation", "source_method", test_cells.geometry.name
]
available = [c for c in base_cols if c in test_cells.columns]
summary = test_cells[available].copy()
summary["grid_1km_id"] = summary["grid_1km_id"].astype(str)

for df in (length_df, elements_df, users_df, contrib_df):
    summary = summary.merge(df, on="grid_1km_id", how="left", validate="one_to_one")

summary["road_length_km"] = summary["road_length_m"] / 1000.0
if "cell_area_m2" in summary.columns:
    summary["road_density_km_per_km2"] = (
        summary["road_length_km"] / (summary["cell_area_m2"] / 1_000_000.0)
    )
summary["ohsome_api_version"] = api_version
summary["oshdb_from"] = oshdb_start
summary["oshdb_to"] = oshdb_end
summary["road_filter"] = ROAD_FILTER
summary["query_run_utc"] = datetime.now(timezone.utc).isoformat()

metric_cols = ["road_length_m","road_element_count","contributors_count","contributions_count"]
qc = {
    "rows_10": len(summary) == 10,
    "grid_id_unique": summary["grid_1km_id"].is_unique,
    "all_metric_values_present": summary[metric_cols].notna().all().all(),
    "all_metric_values_nonnegative": (summary[metric_cols] >= 0).all().all(),
    "all_api_versions_match_metadata": all(
        p.get("apiVersion") == api_version for p in payloads.values()
    ),
}
print(qc)
if not all(qc.values()):
    raise ValueError(f"Final validation failed: {qc}")

summary.to_parquet(SUMMARY_FILE, index=False)
display(summary.drop(columns=[summary.geometry.name], errors="ignore").sort_values("grid_1km_id"))
print("PASS — all four endpoints validated for the 10-cell test sample.")
print("Saved:", SUMMARY_FILE)


{'rows_10': True, 'grid_id_unique': True, 'all_metric_values_present': np.True_, 'all_metric_values_nonnegative': np.True_, 'all_api_versions_match_metadata': True}


,grid_1km_id,cell_area_m2,municipality_fraction,spatial_relation,source_method,road_length_m,road_length_m_timestamp,road_element_count,road_element_count_timestamp,contributors_count,...,contributions_count,contributions_count_from,contributions_count_to,road_length_km,road_density_km_per_km2,ohsome_api_version,oshdb_from,oshdb_to,road_filter,query_run_utc
0,1KME6096N8812,9.999030e+05,1.000000,within,native_1km,2638.23,2026-07-27T09:00:00Z,3.0,2026-07-27T09:00:00Z,2.0,...,5.0,2007-10-08T00:00:00Z,2026-07-27T09:00:00Z,2.63823,2.638486,1.10.4,2007-10-08T00:00:00Z,2026-07-27T09:00Z,highway=* and geometry:line,2026-08-11T00:24:07.201079+00:00
8,1KME6097N8811,9.999116e+05,0.880925,boundary_intersection,native_1km,1464.55,2026-07-27T09:00:00Z,5.0,2026-07-27T09:00:00Z,6.0,...,19.0,2007-10-08T00:00:00Z,2026-07-27T09:00:00Z,1.46455,1.464679,1.10.4,2007-10-08T00:00:00Z,2026-07-27T09:00Z,highway=* and geometry:line,2026-08-11T00:24:07.201079+00:00
1,1KME6097N8812,9.999114e+05,1.000000,within,native_1km,3812.75,2026-07-27T09:00:00Z,5.0,2026-07-27T09:00:00Z,10.0,...,16.0,2007-10-08T00:00:00Z,2026-07-27T09:00:00Z,3.81275,3.813088,1.10.4,2007-10-08T00:00:00Z,2026-07-27T09:00Z,highway=* and geometry:line,2026-08-11T00:24:07.201079+00:00
2,1KME6097N8813,9.999111e+05,1.000000,within,native_1km,0.00,2026-07-27T09:00:00Z,0.0,2026-07-27T09:00:00Z,7.0,...,8.0,2007-10-08T00:00:00Z,2026-07-27T09:00:00Z,0.00000,0.000000,1.10.4,2007-10-08T00:00:00Z,2026-07-27T09:00Z,highway=* and geometry:line,2026-08-11T00:24:07.201079+00:00
3,1KME6098N8812,9.999198e+05,1.000000,within,native_1km,4334.87,2026-07-27T09:00:00Z,6.0,2026-07-27T09:00:00Z,15.0,...,31.0,2007-10-08T00:00:00Z,2026-07-27T09:00:00Z,4.33487,4.335218,1.10.4,2007-10-08T00:00:00Z,2026-07-27T09:00Z,highway=* and geometry:line,2026-08-11T00:24:07.201079+00:00
4,1KME6098N8813,9.999195e+05,1.000000,within,native_1km,3758.68,2026-07-27T09:00:00Z,6.0,2026-07-27T09:00:00Z,16.0,...,38.0,2007-10-08T00:00:00Z,2026-07-27T09:00:00Z,3.75868,3.758982,1.10.4,2007-10-08T00:00:00Z,2026-07-27T09:00Z,highway=* and geometry:line,2026-08-11T00:24:07.201079+00:00
7,1KME6107N8797,1.000001e+06,0.557120,boundary_intersection,aggregated_200m,3626.77,2026-07-27T09:00:00Z,16.0,2026-07-27T09:00:00Z,35.0,...,128.0,2007-10-08T00:00:00Z,2026-07-27T09:00:00Z,3.62677,3.626766,1.10.4,2007-10-08T00:00:00Z,2026-07-27T09:00Z,highway=* and geometry:line,2026-08-11T00:24:07.201079+00:00
6,1KME6118N8826,1.000095e+06,0.152368,boundary_intersection,aggregated_200m,0.00,2026-07-27T09:00:00Z,0.0,2026-07-27T09:00:00Z,3.0,...,3.0,2007-10-08T00:00:00Z,2026-07-27T09:00:00Z,0.00000,0.000000,1.10.4,2007-10-08T00:00:00Z,2026-07-27T09:00Z,highway=* and geometry:line,2026-08-11T00:24:07.201079+00:00
9,1KME6119N8801,1.000110e+06,0.999933,boundary_intersection,aggregated_200m,1359.61,2026-07-27T09:00:00Z,1.0,2026-07-27T09:00:00Z,4.0,...,7.0,2007-10-08T00:00:00Z,2026-07-27T09:00:00Z,1.35961,1.359460,1.10.4,2007-10-08T00:00:00Z,2026-07-27T09:00Z,highway=* and geometry:line,2026-08-11T00:24:07.201079+00:00
5,1KME6120N8829,1.000113e+06,0.000169,boundary_intersection,aggregated_200m,2969.02,2026-07-27T09:00:00Z,7.0,2026-07-27T09:00:00Z,6.0,...,23.0,2007-10-08T00:00:00Z,2026-07-27T09:00:00Z,2.96902,2.968685,1.10.4,2007-10-08T00:00:00Z,2026-07-27T09:00Z,highway=* and geometry:line,2026-08-11T00:24:07.201079+00:00


PASS — all four endpoints validated for the 10-cell test sample.
Saved: /home/runner/work/sotm-br-2026/sotm-br-2026/data/processed/Petropolis/petropolis_ohsome_test_10_cells.parquet


## 8. Scale-up gate

The full 895-cell production query may proceed only if this notebook completes without exception.

Validated response contracts are endpoint-specific:
- element length/count: `groupByResult[].result[].timestamp + value`;
- user count: grouped interval result with an accepted temporal representation;
- contribution count: grouped interval result, normally `fromTimestamp + toTimestamp + value`.

Raw success and error envelopes are retained so that a future API patch can be diagnosed without losing the server response.
